# ECS 171 Group Project Team 9

## Introduction

Heart disease is one of the leading causes of death worldwide. Early prediction of heart disease risk can support timely medical intervention and lifestyle changes. The goal of this project is to develop and evaluate machine learning models that predict whether a patient is at risk of heart disease based on clinical and demographic attributes.

This is formulated as a binary classification problem, where the output indicates the presence or absence of heart disease.

## Dataset Description

Dataset: UCI Heart Disease Dataset

Size: ~300 instances
Attributes: ~13 features
Target variable: Presence of heart disease (0 = No, 1 = Yes)

## Exploratory Data Analysis (EDA)

### Data Input

In [ ]:
from ucimlrepo import fetch_ucirepo

heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

print(heart_disease.metadata)
print(heart_disease.variables)

In [ ]:
X = heart_disease.data.features.copy()
y = heart_disease.data.targets.copy()
X.head()

In [ ]:
y.head()

### Data Cleaning

#### Finding Missing Values and Replace it with NaN

In [ ]:
import numpy as np
import pandas as pd

X = X.replace('?', np.nan)
X.isna().sum()

#### Turning Data Types to Numeric

In [ ]:
num_cols = ['age','trestbps','chol','thalach','oldpeak','ca']
X[num_cols] = X[num_cols].apply(pd.to_numeric)

cat_cols = ['sex','cp','fbs','restecg','exang','slope','thal']
X[cat_cols] = X[cat_cols].apply(pd.to_numeric)

X.info()

#### Turn NaN into numbers

In [ ]:
X['ca'] = X['ca'].fillna(X['ca'].median())
X['thal'] = X['thal'].fillna(X['thal'].mode()[0])

X.isna().sum()

#### Outlier Detection

In [ ]:
def detect_outliers(df, columns):
    outlier_indices = []
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower) | (df[col] > upper)].index
        outlier_indices.extend(outliers)

    return list(set(outlier_indices))


In [ ]:
outlier_cols = ['age','trestbps','chol','thalach','oldpeak']
outliers = detect_outliers(X, outlier_cols)

len(outliers)

#### Cap Outlier

In [ ]:
def cap_outliers(df, columns):
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        df[col] = np.where(df[col] < lower, lower, df[col])
        df[col] = np.where(df[col] > upper, upper, df[col])
        
    return df

X = cap_outliers(X, outlier_cols)

#### Change Target Value from severity of hear disease to if the patient has heart disease or not

In [ ]:
y = (y['num'] > 0).astype(int)

In [ ]:
y.head()

#### Encode Categorical Variables - One-Hot Encoding

Ex. If a patient has: cp = 3   (non-anginal pain)

it becomes:
cp_2	cp_3	cp_4
0	1	0

Meaning:

The patient is not type 2

The patient is type 3

The patient is not type 4

In [ ]:
X = pd.get_dummies(X, columns=['cp','restecg','slope','thal'], drop_first=True)
X.head()

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)